## Preliminary steps

In [1]:
import getpass
import os

import repl
from dotenv.variables import Literal
from jedi.inference.recursion import recursion_limit
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage

def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Provide {var} ")

_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("TAVILY_API_KEY")

In [17]:
!pip install -U langgraph langchain_community langchain-tavily langchain-experimental langchain_openai

  Using cached langgraph-1.1.3-py3-none-any.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## All imports

In [2]:
from langchain_core.tools import tool
from typing import Annotated, List, Optional, Dict, Literal, TypedDict
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Define tools

In [10]:
@tool
def scrape_webpages(urls: List[str]) -> str:
    """User requests to scrape and bs4 to scrape the provided web pages for detailed information."""
    loader = WebBaseLoader(urls)
    doc = loader.load()
    return "\n\n".join(
        [
            f"<Document name={doc.metadata.get("title", "")}>\n{doc.page_content}\n</Document>"
        ]
    )

@tool
def create_outline(
    points: Annotated[List[str], "List of main points or sections"],
    file_name: Annotated[str, "File path to save the outline"]
) -> Annotated[str, "Path of the saved outline file"]:
    """Create and save an outline"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"

@tool
def read_document(
    file_name: Annotated[str, "File path of the document to read"],
    start: Annotated[Optional[int], "The start line. Default is 0"] = None,
    end: Annotated[Optional[int], "The start line. Default is None"] = None
):
    """Read the specific document"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "r") as file:
        lines = file.readlines()
    if start is None:
        start = 0
    return "\n".join(lines[start:end])

@tool
def write_document(
    content: Annotated[str, "Text content to be written to the document"],
    file_name: Annotated[str, "File path to save the document"]
):
    """Create and save a text document"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        file.write(content)
    return f"Document saved to {file_name}"

@tool
def edit_document(
    file_name: Annotated[str, "File path of the document to edit"],
    inserts: Annotated[Dict[int, str], "Dictionary where key is the line number and value is the text to be inserted at the line"]
):
    """Edit a document by inserting text at specified line numbers"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        lines = file.readlines()

    sorted_inserts = sorted(inserts.items())

    for line_num, text in sorted_inserts:
        if 1 <= line_num <= len(lines) + 1:
            lines.insert(line_num - 1, text + "\n")
        else:
           return f"Error: Line number {line_num} is out of range"

    with open(file_to_use, "w") as file:
        file.writelines(lines)

    return f"Document edited and saved to {file_name}"

def python_repl_tool(
        code: Annotated[str, "The python code to execute to generate the chart"],
):
    """Use this to execute python code. If want to see output of any value, should print it with `print(...)`. This is visible to the user"""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Error executing code: {e}"

    return f"Code executed.\nResult:\n````\n{code}\n```\n Stdout:{result}"

## Supervisor agent

In [11]:
class State(MessagesState):
    next: str

def make_supervisor_node(llm: BaseChatModel, members: List[str]):
    Options = ["FINISH"] + members
    system_prompt = {
        f"""You are a supervisor agent tasked with managing a conversation between the following workers: {members}. Given the following user requests, respond with the worker to act next. Each worker will perform a task and respond with their results and status. When finished, respond with FINISH"""
    }

    class Router[TypedDict]:
        next: Literal[tuple(Options)]

    def supervisor(state: State) -> Command:
        messages = [
            {"role": "system", "content": system_prompt}
        ] + state["messages"]

        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        if goto == "FINISH":
            goto = END

        return Command(goto=goto, update={"next": goto})
    return supervisor

NameError: name 'BaseChatModel' is not defined

## Agent teams

### Research team

In [48]:
llm = ChatOpenAI(model="gpt-4o")
tavily_tool = TavilySearch(max_results=3)

search_agent = create_react_agent(llm, tools=[tavily_tool])

def search_node(state: State) -> Command[Literal["supervisor"]]:
    result = search_agent.invoke(state)
    return Command(
        update = {
            "messages": [HumanMessage(content=result["messages"][-1].content, name="search")]
        },
        goto = "supervisor"
    )

web_scrapping_agent = create_react_agent(llm, tools=[scrape_webpages])

def web_scrapper_node(state: State) -> Command[Literal["supervisor"]]:
    result = scrape_webpages.invoke(state)
    return Command(
        update = {
            "messages": [HumanMessage(content=result["messages"][-1].content, name="web_scrapper")]
        },
        goto = "supervisor"
    )

research_supervisor_node = make_supervisor_node(llm, ["search", "web_scrapper"])

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [19]:
research_builder = StateGraph(State)
research_builder.add_node("supervisor", research_supervisor_node)
research_builder.add_node("search", search_node)
research_builder.add_node("web_scrapper", web_scrapper_node)

research_builder.set_entry_point("supervisor")
research_graph = research_builder.compile()

### Writing team

In [20]:
doc_writer_agent = create_react_agent(
    llm,
    tools = [write_document, read_document, edit_document],
    prompt = "You can read, write and edit documents based on note taker's outlines. Don't ask followup questions."
)

def doc_writer_node(state: State) -> Command[Literal["supervisor"]]:
    result = doc_writer_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="doc_writer")]
        },
        goto = "supervisor"
    )

note_taker_agent = create_react_agent(
    llm,
    tools = [create_outline],
    prompt = "You can read and create outlines for the document writer. Don't ask followup questions."
)

def note_taker_node(state: State) -> Command[Literal["supervisor"]]:
    result = note_taker_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="note_taker")]
        },
        goto = "supervisor"
    )

chart_generator_agent = create_react_agent(
    llm,
    tools = [read_document, python_repl_tool],
)

def chart_generator_node(state: State) -> Command[Literal["supervisor"]]:
    result = chart_generator_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="chart_generator")]
        },
        goto = "supervisor"
    )

doc_writer_supervisor_node = make_supervisor_node(llm, ["note_taker", "doc_writer", "chart_generator"])


C:\Users\LOQ\AppData\Local\Temp\ipykernel_12252\2584305033.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  doc_writer_agent = create_react_agent(
C:\Users\LOQ\AppData\Local\Temp\ipykernel_12252\2584305033.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  note_taker_agent = create_react_agent(
C:\Users\LOQ\AppData\Local\Temp\ipykernel_12252\2584305033.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  chart_generator_agent = create_react_agent(


In [21]:
writing_builder = StateGraph(State)
writing_builder.add_node("supervisor", doc_writer_supervisor_node)
writing_builder.add_node("doc_writer", doc_writer_node)
writing_builder.add_node("note_taker", note_taker_node)
writing_builder.add_node("chart_generator", chart_generator_node)

writing_builder.set_entry_point("supervisor")
writing_graph = writing_builder.compile()

In [12]:
for s in writing_graph.stream(
    {
        "messages": [
            (
                "user",
                "Write an outline for a poem about cats and after that write the poem itself and store it"
            )
        ]
    },
    {"recursion_limit": 30}
):
    print(s)
    print("-" * 10)

NameError: name 'writing_graph' is not defined

## End to end graph

In [1]:
teams_supervisor_node = make_supervisor_node(llm, ["research", "writing"])

def call_research_team(state: State) -> Command[Literal["supervisor"]]:
    response = research_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update = {
            "messages": [
                HumanMessage(content=response["messages"][-1].content, name="research")
            ]
        },
        goto = "supervisor"
    )

def call_writing_team(state: State) -> Command[Literal["supervisor"]]:
    response = writing_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update = {
            "messages": [
                HumanMessage(content=response["messages"][-1].content, name="writing")
            ]
        },
        goto = "supervisor"
    )

super_builder = StateGraph(State)
super_builder.add_node("supervisor", teams_supervisor_node)
super_builder.add_node("research", call_research_team)
super_builder.add_node("writing", call_writing_team)

super_builder.set_entry_point("supervisor")
super_graph = super_builder.compile()


NameError: name 'make_supervisor_node' is not defined

In [2]:
from IPython.display import Image, display

display(Image(super_graph.get_graph().draw_mermaid_png()))

NameError: name 'super_graph' is not defined

In [3]:
for s in super_graph.stream(
    {
        "messages": [
            (
                "user",
                "Write a poem about cats. First research about cats and then write the poem based on your research. Make sure to include interesting facts about cats in the poem."
            )
        ]
    },
    {"recursion_limit": 100}
):
    print(s)
    print("-" * 10)

NameError: name 'super_graph' is not defined